In [1]:
import pickle
import numpy as np
from transformers import BartTokenizer, BartForConditionalGeneration
from data import ZuCo_dataset
from torch.utils.data import DataLoader

whole_dataset_dicts = []

dataset_path_task1 = '/home/saul_park/workspace/code/Baseline/dataset/ZuCo/task1-SR/pickle/task1-SR-dataset.pickle'
with open(dataset_path_task1, 'rb') as handle:
    whole_dataset_dicts.append(pickle.load(handle))

dataset_path_task2 = '/home/saul_park/workspace/code/Baseline/dataset/ZuCo/task2-NR/pickle/task2-NR-dataset.pickle'
with open(dataset_path_task2, 'rb') as handle:
    whole_dataset_dicts.append(pickle.load(handle))


dataset_path_taskNRv2 = '/home/saul_park/workspace/code/Baseline/dataset/ZuCo/task2-NR-2.0/pickle/task2-NR-2.0-dataset.pickle'

with open(dataset_path_taskNRv2, 'rb') as handle:
    whole_dataset_dicts.append(pickle.load(handle))

tokenizer = BartTokenizer.from_pretrained('facebook/bart-large')
subject_choice = 'ALL'
batch_size = 32
eeg_type_choice = 'GD'
bands_choice = ['_t1','_t2','_a1','_a2','_b1','_b2','_g1','_g2']
dataset_setting = 'unique_sent'
train_input = 'EEG'

# train dataset
train_set = ZuCo_dataset(whole_dataset_dicts, 'train', tokenizer, subject = subject_choice, eeg_type = eeg_type_choice, bands = bands_choice, setting = dataset_setting, test_input=train_input)
# dev dataset
dev_set = ZuCo_dataset(whole_dataset_dicts, 'dev', tokenizer, subject = subject_choice, eeg_type = eeg_type_choice, bands = bands_choice, setting = dataset_setting, test_input=train_input)
# test dataset
test_set = ZuCo_dataset(whole_dataset_dicts, 'test', tokenizer, subject = subject_choice, eeg_type = eeg_type_choice, bands = bands_choice, setting = dataset_setting)
dataset_sizes = {'train': len(train_set), 'dev': len(dev_set)}
print('[INFO]train_set size: ', len(train_set))
print('[INFO]dev_set size: ', len(dev_set))
print('[INFO]test_set size: ', len(test_set))

# train dataloader
train_dataloader = DataLoader(train_set, batch_size = batch_size, shuffle=True, num_workers=4)
# dev dataloader
val_dataloader = DataLoader(dev_set, batch_size = 1, shuffle=False, num_workers=4)
test_dataloader = DataLoader(test_set, batch_size = 1, shuffle=False, num_workers=4)
# dataloaders
dataloaders = {'train':train_dataloader, 'dev':val_dataloader, 'test':test_dataloader}

for input_embeddings, seq_len, input_masks, input_mask_invert, target_ids, target_mask, rawEEG in dataloaders['test']:
    break

[INFO]loading 3 task datasets
[INFO]using subjects:  ['ZAB', 'ZDM', 'ZDN', 'ZGW', 'ZJM', 'ZJN', 'ZJS', 'ZKB', 'ZKH', 'ZKW', 'ZMG', 'ZPH']
train divider = 320
dev divider = 360
[INFO]initializing a train set...
++ adding task to dataset, now we have: 3609
[INFO]using subjects:  ['ZAB', 'ZDM', 'ZDN', 'ZGW', 'ZJM', 'ZJN', 'ZJS', 'ZKB', 'ZKH', 'ZKW', 'ZMG', 'ZPH']
train divider = 240
dev divider = 270
[INFO]initializing a train set...
++ adding task to dataset, now we have: 6254
[INFO]using subjects:  ['YDR', 'YFR', 'YFS', 'YAC', 'YDG', 'YHS', 'YMD', 'YLS', 'YRH', 'YRK', 'YSD', 'YRP', 'YSL', 'YTL', 'YAG', 'YAK', 'YIS', 'YMS']
train divider = 279
dev divider = 313
[INFO]initializing a train set...
++ adding task to dataset, now we have: 10710
[INFO]input tensor size: torch.Size([56, 840])

[INFO]loading 3 task datasets
[INFO]using subjects:  ['ZAB', 'ZDM', 'ZDN', 'ZGW', 'ZJM', 'ZJN', 'ZJS', 'ZKB', 'ZKH', 'ZKW', 'ZMG', 'ZPH']
train divider = 320
dev divider = 360
[INFO]initializing a dev set

In [2]:
from ldm.autoencoder import AutoencoderKL
from ldm.openaimodel import UNetModel
from model_decoding import LDMTranslator, pretrainedmodule
import torch

vocab_size = tokenizer.vocab_size
pretrained = BartForConditionalGeneration.from_pretrained('facebook/bart-large')

eeg_encoder = AutoencoderKL(
    embed_dim=56,
    # embed_dim=1024,
    monitor="val/rec_loss",
    ddconfig={
        "double_z": True,
        "z_channels": 58, 
        "resolution": 256,
        "in_channels": 56,
        "out_ch": 56,
        # "in_channels": 840,
        # "out_ch": 840,
        "ch": 128,
        "ch_mult": [1, 2, 4, 4],
        "num_res_blocks": 2,
        "attn_resolutions": [],
        "dropout": 0.0
    },
    lossconfig={
        "target": "torch.nn.Identity"
    },
    vocab_size = vocab_size,
    latent_dim=1024,
    type = 'eeg'
).to('cpu')

unet = UNetModel(
    image_size=512,
    dims = 1, 
    # in_channels=decoder_embedding_size, 
    # out_channels=decoder_embedding_size, 
    in_channels=56, 
    out_channels=56, 
    model_channels=320, 
    attention_resolutions=[4, 2, 1], 
    num_res_blocks=2, 
    channel_mult=[1, 2, 4, 4], 
    num_heads=8, 
    use_spatial_transformer=True, 
    transformer_depth=1, 
    context_dim=512, 
    use_checkpoint=False, 
    legacy=False
    ).to('cpu')

path = 'checkpoints/best/pretrain_autoencoder_unet_no_context_noise_to_eeg_task1_task2_taskNRv2_2steptraining_b32_100_50_2e-05_2e-05.pt'
pretrained_module = pretrainedmodule(eeg_encoder, unet, pretrained, 'cpu').to('cpu')

model = LDMTranslator(pretrained=pretrained_module)

state_dict = torch.load(path)
# new_state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
model.load_state_dict(state_dict)

making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 58, 32, 32) = 59392 dimensions.
making attention of type 'vanilla' with 512 in_channels


/home/saul_park/workspace/code/pretrain_eeg_encoder/ldm/diffusion_module.py:39: RuntimeWarning: divide by zero encountered in true_divide
  lvlb_weights = betas ** 2 / (


<All keys matched successfully>

In [3]:
input_embeddings_batch = input_embeddings.to('cpu').float() # B, 56, 840
# input_embeddings_batch = sent_level_EEG.unsqueeze(1).to(device).float()
input_masks_batch = input_masks.to('cpu') # B, 56
target_ids_batch = target_ids.to('cpu') # B, 56
input_mask_invert_batch = input_mask_invert.to('cpu') # B, 56

### IJCAI

##### TF

In [74]:
def remove_text_after_token(text, token='</s>'):
    # 특정 토큰 이후의 텍스트를 찾아 제거
    token_index = text.find(token)
    if token_index != -1:  # 토큰이 발견된 경우
        return text[:token_index]  # 토큰 이전까지의 텍스트 반환
    return text  # 토큰이 없으면 원본 텍스트 반환


target_tokens_list = []
target_string_list = []
pred_tokens_list = []
pred_string_list = []
pred_tokens_list_previous = []
pred_string_list_previous = []


target_tokens = tokenizer.convert_ids_to_tokens(target_ids_batch[0].tolist(), skip_special_tokens = True)

target_string = tokenizer.decode(target_ids_batch[0], skip_special_tokens = True)
# print('target ids tensor:',target_ids_batch[0])
# print('target ids:',target_ids_batch[0].tolist())
# print('target tokens:',target_tokens)
# print('target string:',target_strininvert.to(device) # B, 56


# add to list for later calculate bleu metric
target_tokens_list.append([target_tokens])
target_string_list.append(target_string)
print(f'target tokens: {target_tokens_list}\n')
print(f'target string: {target_string_list}\n')
context = target_ids_batch.clone()
"""replace padding ids in target_ids with -100"""
new_target_ids = torch.where(
    target_ids_batch == tokenizer.pad_token_id,  # 조건
    torch.tensor(-100, dtype=target_ids_batch.dtype),  # 조건이 True일 때 대체 값
    target_ids_batch  # 조건이 False일 때 원래 값
)

# target_ids_batch_label = target_ids_batch.clone().detach()
# target_ids_batch_label[target_ids_batch_label == tokenizer.pad_token_id] = -100

# Original code 
# seq2seqLMoutput,_ = model(input_embeddings_batch, input_masks_batch, input_mask_invert_batch, target_ids_batch, context) # (batch, time, n_class)
seq2seqLMoutput = model.evaluation(input_embeddings_batch, input_masks_batch, input_mask_invert_batch, new_target_ids, context) # (batch, time, n_class)
logits_previous = seq2seqLMoutput.logits
probs_previous = logits_previous[0].softmax(dim = 1)
values_previous, predictions_previous = probs_previous.topk(1)
predictions_previous = torch.squeeze(predictions_previous)
print(f'predictions previous: {tokenizer.decode(predictions_previous)}\n')

predicted_string_previous = remove_text_after_token(tokenizer.decode(predictions_previous).split('</s></s>')[0].replace('<s>',''))

predictions_previous = predictions_previous.tolist()
truncated_prediction_previous = []
for t in predictions_previous:
    if t != tokenizer.eos_token_id:
        truncated_prediction_previous.append(t)
    else:
        break
print(f'predictions previous: {truncated_prediction_previous}\n')
pred_tokens_previous = tokenizer.convert_ids_to_tokens(truncated_prediction_previous, skip_special_tokens = True)
pred_tokens_list_previous.append(pred_tokens_previous)
pred_string_list_previous.append(predicted_string_previous)
print(f'predicted tokens previous: {pred_tokens_list_previous}\n')
print(f'predicted string previous: {pred_string_list_previous}\n')

target tokens: [[['Everything', 'Ġits', 'Ġtitle', 'Ġimplies', ',', 'Ġa', 'Ġstandard', '-', 'issue', 'Ġcrime', 'Ġdrama', 'Ġspat', 'Ġout', 'Ġfrom', 'Ġthe', 'ĠTin', 'sel', 'town', 'Ġassembly', 'Ġline', '.']]]

target string: ['Everything its title implies, a standard-issue crime drama spat out from the Tinseltown assembly line.']

predictions previous: <s><s> about about implies is from man,issue, drama. between of the depthsseltowns..</s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s>

predictions previous: [0, 0, 59, 59, 19659, 16, 31, 313, 6, 25268, 6, 4149, 4, 227, 9, 5, 24297, 5317, 7171, 29, 4, 4]

predicted tokens previous: [['Ġabout', 'Ġabout', 'Ġimplies', 'Ġis', 'Ġfrom', 'Ġman', ',', 'issue', ',', 'Ġdrama', '.', 'Ġbetween', 'Ġof', 'Ġthe', 'Ġdepths', 'sel', 'town', 's', '.', '.']]

predicted string previous: [' about about implies is from man,issue, drama. between of the depthsseltowns..']



In [5]:
from nltk.translate.bleu_score import corpus_bleu
weights_list = [(1.0,),(0.5,0.5),(1./3.,1./3.,1./3.),(0.25,0.25,0.25,0.25)]

for weight in weights_list:
    # print('weight:',weight)
    corpus_bleu_score_previous = corpus_bleu(target_tokens_list, pred_tokens_list_previous, weights = weight)
    print(f'corpus BLEU-{len(list(weight))} score with tf:', corpus_bleu_score_previous)

corpus BLEU-1 score with tf: 0.42805324102532133
corpus BLEU-2 score with tf: 0.1463911196106228
corpus BLEU-3 score with tf: 7.68339317180261e-104
corpus BLEU-4 score with tf: 5.566371910254372e-155


/home/saul_park/miniconda3/envs/sejin/lib/python3.12/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/home/saul_park/miniconda3/envs/sejin/lib/python3.12/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)


In [6]:
def remove_text_after_token(text, token='</s>'):
    # 특정 토큰 이후의 텍스트를 찾아 제거
    token_index = text.find(token)
    if token_index != -1:  # 토큰이 발견된 경우
        return text[:token_index]  # 토큰 이전까지의 텍스트 반환
    return text  # 토큰이 없으면 원본 텍스트 반환
def ids_to_readable_tokens(batch_ids, tokenizer):
    """
    Byte-Level BPE 토큰 ID → 사람이 읽기 좋은 토큰 문자열
    (선행 공백을 보존하되 Ġ 대신 실제 공백으로 치환)
    """
    for ids in batch_ids:
        raw_tokens = tokenizer.convert_ids_to_tokens(
            ids, skip_special_tokens=True        # 특수 토큰 제거
        )
        readable = [tok.replace("Ġ", " ") for tok in raw_tokens]
    return readable


target_tokens_list = []
target_string_list = []
pred_tokens_list = []
pred_string_list = []
pred_tokens_list_previous = []
pred_string_list_previous = []


# target_tokens = tokenizer.batch_decode(target_ids_batch, skip_special_tokens = True)
target_tokens = ids_to_readable_tokens(target_ids_batch, tokenizer)

target_string = tokenizer.decode(target_ids_batch[0], skip_special_tokens = True)
# print('target ids tensor:',target_ids_batch[0])
# print('target ids:',target_ids_batch[0].tolist())
# print('target tokens:',target_tokens)
# print('target string:',target_strininvert.to(device) # B, 56


# add to list for later calculate bleu metric
target_tokens_list.append([target_tokens])
target_string_list.append(target_string)
print(f'target tokens: {target_tokens_list}\n')
# print(f'target string: {target_string_list}\n')
context = target_ids_batch.clone()
"""replace padding ids in target_ids with -100"""
new_target_ids = torch.where(
    target_ids_batch == tokenizer.pad_token_id,  # 조건
    torch.tensor(-100, dtype=target_ids_batch.dtype),  # 조건이 True일 때 대체 값
    target_ids_batch  # 조건이 False일 때 원래 값
)

# target_ids_batch_label = target_ids_batch.clone().detach()
# target_ids_batch_label[target_ids_batch_label == tokenizer.pad_token_id] = -100

# Original code 
# seq2seqLMoutput,_ = model(input_embeddings_batch, input_masks_batch, input_mask_invert_batch, target_ids_batch, context) # (batch, time, n_class)
seq2seqLMoutput = model.evaluation(input_embeddings_batch, input_masks_batch, input_mask_invert_batch, new_target_ids, context) # (batch, time, n_class)
logits_previous = seq2seqLMoutput.logits
probs_previous = logits_previous[0].softmax(dim = 1)
values_previous, predictions_previous = probs_previous.topk(1)
predictions_previous = torch.squeeze(predictions_previous)
predicted_string_previous = remove_text_after_token(tokenizer.decode(predictions_previous).split('</s></s>')[0].replace('<s>',''))

predictions_previous = predictions_previous.tolist()
truncated_prediction_previous = []
for t in predictions_previous:
    if t != tokenizer.eos_token_id:
        truncated_prediction_previous.append(t)
    else:
        break
# pred_tokens_previous = tokenizer.batch_decode(truncated_prediction_previous, skip_special_tokens = True)
pred_tokens_previous = ids_to_readable_tokens([truncated_prediction_previous], tokenizer)
pred_tokens_list_previous.append(pred_tokens_previous)
pred_string_list_previous.append(predicted_string_previous)
print(f'predicted tokens previous: {pred_tokens_list_previous}\n')
print(f'predicted string previous: {pred_string_list_previous}\n')

target tokens: [[['Everything', ' its', ' title', ' implies', ',', ' a', ' standard', '-', 'issue', ' crime', ' drama', ' spat', ' out', ' from', ' the', ' Tin', 'sel', 'town', ' assembly', ' line', '.']]]

predicted tokens previous: [[' about', ' about', ' implies', ' is', ' from', ' man', ',', 'issue', ',', ' drama', '.', ' between', ' of', ' the', ' depths', 'sel', 'town', 's', '.', '.']]

predicted string previous: [' about about implies is from man,issue, drama. between of the depthsseltowns..']



##### evaluation

In [7]:
from nltk.translate.bleu_score import corpus_bleu
weights_list = [(1.0,),(0.5,0.5),(1./3.,1./3.,1./3.),(0.25,0.25,0.25,0.25)]

for weight in weights_list:
    # print('weight:',weight)
    corpus_bleu_score_previous = corpus_bleu(target_tokens_list, pred_tokens_list_previous, weights = weight)
    print(f'corpus BLEU-{len(list(weight))} score with tf:', corpus_bleu_score_previous)


from rouge import Rouge
import evaluate
from evaluate import load

metric = evaluate.load("sacrebleu")
cer_metric = load("cer")
wer_metric = load("wer")
rouge = Rouge()
try:
    rouge_scores_previous = rouge.get_scores(pred_string_list_previous, target_string_list, avg = True, ignore_empty=True)
except ValueError as e:
    rouge_scores_previous = 'Hypothesis is empty'

print(f'rouge scores previous: {rouge_scores_previous}\n')

wer_scores_previous = wer_metric.compute(predictions=pred_string_list_previous, references=target_string_list)

print("WER score with tf:", wer_scores_previous)

cer_scores_previous = cer_metric.compute(predictions=pred_string_list_previous, references=target_string_list)
print("CER score with tf:", cer_scores_previous)

corpus BLEU-1 score with tf: 0.42805324102532133
corpus BLEU-2 score with tf: 0.1463911196106228
corpus BLEU-3 score with tf: 7.68339317180261e-104
corpus BLEU-4 score with tf: 5.566371910254372e-155
rouge scores previous: {'rouge-1': {'r': 0.2, 'p': 0.3, 'f': 0.23999999520000007}, 'rouge-2': {'r': 0.0, 'p': 0.0, 'f': 0.0}, 'rouge-l': {'r': 0.13333333333333333, 'p': 0.2, 'f': 0.15999999520000013}}

WER score with tf: 0.9333333333333333
CER score with tf: 0.6274509803921569


##### without TF

In [11]:
def remove_text_after_token(text, token='</s>'):
    # 특정 토큰 이후의 텍스트를 찾아 제거
    token_index = text.find(token)
    if token_index != -1:  # 토큰이 발견된 경우
        return text[:token_index]  # 토큰 이전까지의 텍스트 반환
    return text  # 토큰이 없으면 원본 텍스트 반환


target_tokens_list = []
target_string_list = []
pred_tokens_list = []
pred_string_list = []
pred_tokens_list_previous = []
pred_string_list_previous = []


target_tokens = tokenizer.convert_ids_to_tokens(target_ids_batch[0].tolist(), skip_special_tokens = True)

target_string = tokenizer.decode(target_ids_batch[0], skip_special_tokens = True)
# print('target ids tensor:',target_ids_batch[0])
# print('target ids:',target_ids_batch[0].tolist())
# print('target tokens:',target_tokens)
# print('target string:',target_strininvert.to(device) # B, 56


# add to list for later calculate bleu metric
target_tokens_list.append([target_tokens])
target_string_list.append(target_string)
print(f'target tokens: {target_tokens_list}\n')
print(f'target string: {target_string_list}\n')
context = target_ids_batch.clone()
"""replace padding ids in target_ids with -100"""
new_target_ids = torch.where(
    target_ids_batch == tokenizer.pad_token_id,  # 조건
    torch.tensor(-100, dtype=target_ids_batch.dtype),  # 조건이 True일 때 대체 값
    target_ids_batch  # 조건이 False일 때 원래 값
)
dummy_decoder_input_ids = torch.tensor([[tokenizer.pad_token_id]]).to('cpu') # B, 1
predictions=model.generate(input_embeddings_batch, input_masks_batch, input_mask_invert_batch, new_target_ids,dummy_decoder_input_ids,
                            max_length=56,
                            num_beams=5,
                            do_sample=True,
                            repetition_penalty= 5.0,
                            no_repeat_ngram_size = 2,
                            # num_beams=5,encoder_no_repeat_ngram_size =1,
                            # do_sample=True, top_k=15,temperature=0.5,num_return_sequences=5,
                            # early_stopping=True
                            )

predicted_string=tokenizer.batch_decode(predictions, skip_special_tokens=True)[0]
# predicted_string=predicted_string.squeeze()

predictions=tokenizer.encode(predicted_string)
# print('predicted string:',predicted_string)
print(f'predicted string: {predicted_string}\n')
print(f'predicted:{predictions}\n')

truncated_prediction = []
for t in predictions:
    if t != tokenizer.eos_token_id:
        truncated_prediction.append(t)
    else:
        break

pred_tokens = tokenizer.convert_ids_to_tokens(truncated_prediction, skip_special_tokens = True)
# print('predicted tokens:',pred_tokens)
pred_tokens_list.append(pred_tokens)
pred_string_list.append(predicted_string)

print(f'predicted tokens: {pred_tokens_list}\n')
print(f'predicted string: {pred_string_list}\n')

target tokens: [[['Everything', 'Ġits', 'Ġtitle', 'Ġimplies', ',', 'Ġa', 'Ġstandard', '-', 'issue', 'Ġcrime', 'Ġdrama', 'Ġspat', 'Ġout', 'Ġfrom', 'Ġthe', 'ĠTin', 'sel', 'town', 'Ġassembly', 'Ġline', '.']]]

target string: ['Everything its title implies, a standard-issue crime drama spat out from the Tinseltown assembly line.']

predicted string: is a film that is based on the popular American television series of the same name.

predicted:[0, 354, 10, 822, 14, 16, 716, 15, 5, 1406, 470, 2384, 651, 9, 5, 276, 766, 4, 2]

predicted tokens: [['is', 'Ġa', 'Ġfilm', 'Ġthat', 'Ġis', 'Ġbased', 'Ġon', 'Ġthe', 'Ġpopular', 'ĠAmerican', 'Ġtelevision', 'Ġseries', 'Ġof', 'Ġthe', 'Ġsame', 'Ġname', '.']]

predicted string: ['is a film that is based on the popular American television series of the same name.']



##### evaluation

In [12]:
weights_list = [(1.0,),(0.5,0.5),(1./3.,1./3.,1./3.),(0.25,0.25,0.25,0.25)]
corpus_bleu_scores = []
corpus_bleu_scores_previous = []
for weight in weights_list:
    # print('weight:',weight)
    corpus_bleu_score = corpus_bleu(target_tokens_list, pred_tokens_list, weights = weight)
    corpus_bleu_scores.append(corpus_bleu_score)
    print(f'corpus BLEU-{len(list(weight))} score:', corpus_bleu_score)

corpus BLEU-1 score: 0.1394714758202644
corpus BLEU-2 score: 4.952465347544888e-155
corpus BLEU-3 score: 3.506981394232103e-206
corpus BLEU-4 score: 9.33231673128302e-232


/home/saul_park/miniconda3/envs/sejin/lib/python3.12/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/home/saul_park/miniconda3/envs/sejin/lib/python3.12/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/home/saul_park/miniconda3/envs/sejin/lib/python3.12/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contain

### Original

##### TF

In [13]:
from torchmetrics.text import BLEUScore
from torchmetrics.text import WordErrorRate
from torchmetrics.functional.text.rouge import rouge_score


def compute_metrics(preds, labels):
    result={}
    wer = WordErrorRate()
    result['wer']=wer(preds,labels).item()
    rouge_result=compute_rouge(preds,labels)
    for k,v in rouge_result.items():
        result[k]=v.item()
    labels=[[label] for i,label in enumerate(labels)]
    for i in range(1,5):
        bleu=BLEUScore(n_gram=i, smooth=True)
        result[f'bleu-{i}']=bleu(preds,labels).item()
    return result


def compute_rouge(preds, labels):

    metrics={}
    for decoded_label, decoded_pred in zip(labels, preds):
        metric=rouge_score(decoded_pred,decoded_label)
        for key in metric.keys():
            metrics[key]=metrics.get(key, 0) + metric[key]
    for key in metrics.keys():
        metrics[key]=metrics[key]/len(labels)*100

    return metrics

In [14]:
target_string = tokenizer.batch_decode(target_ids_batch, skip_special_tokens=True)
target_string_list = []
pred_string_list = []
target_string_list.extend(target_string)

"""replace padding ids in target_ids with -100"""
new_target_ids = torch.where(
    target_ids_batch == tokenizer.pad_token_id,  # 조건
    torch.tensor(-100, dtype=target_ids_batch.dtype),  # 조건이 True일 때 대체 값
    target_ids_batch  # 조건이 False일 때 원래 값
)

seq2seqLMoutput = model.evaluation(input_embeddings_batch, input_masks_batch, input_mask_invert_batch, new_target_ids, context) # (batch, time, n_class)
logits = seq2seqLMoutput.logits  # bs*seq_len*voc_sz
probs = logits.softmax(dim=-1)
values, predictions = probs.topk(1)
predictions = torch.squeeze(predictions, dim=-1)

predicted_string = tokenizer.batch_decode(predictions, skip_special_tokens=True, )





pred_string_list.extend(predicted_string)

print(f'Predicted: {pred_string_list}\n')
print(f'True: {target_string_list}\n')

metrics_results=compute_metrics(pred_string_list,target_string_list)
metrics_results

Predicted: [' about about implies is from man,issue, drama. between of the depthsseltowns..']

True: ['Everything its title implies, a standard-issue crime drama spat out from the Tinseltown assembly line.']



{'wer': 0.9333333373069763,
 'rouge1_fmeasure': 35.71428680419922,
 'rouge1_precision': 41.666664123535156,
 'rouge1_recall': 31.25,
 'rouge2_fmeasure': 0.0,
 'rouge2_precision': 0.0,
 'rouge2_recall': 0.0,
 'rougeL_fmeasure': 28.571430206298828,
 'rougeL_precision': 33.333335876464844,
 'rougeL_recall': 25.0,
 'rougeLsum_fmeasure': 28.571430206298828,
 'rougeLsum_precision': 33.333335876464844,
 'rougeLsum_recall': 25.0,
 'bleu-1': 0.12638981640338898,
 'bleu-2': 0.0,
 'bleu-3': 0.0,
 'bleu-4': 0.0}

##### without TF

In [15]:
predictions=model.generate(input_embeddings_batch, input_masks_batch, input_mask_invert_batch, new_target_ids,
                        max_length=56,
                        num_beams=5,
                        do_sample=True,
                        repetition_penalty= 5.0,
                        no_repeat_ngram_size = 2,
                        # num_beams=5,encoder_no_repeat_ngram_size =1,
                        # do_sample=True, top_k=15,temperature=0.5,num_return_sequences=5,
                        # early_stopping=True
                        )

predicted_string = tokenizer.batch_decode(predictions, skip_special_tokens=True, )

print(f'Predicted: {predicted_string}\n')
print(f'True: {target_string}\n')

pred_string_list = []
pred_string_list.extend(predicted_string)
metrics_results=compute_metrics(pred_string_list,target_string_list)
metrics_results

TypeError: LDMTranslator.generate() missing 1 required positional argument: 'dummy_decoder_input_ids'

### evaluate.load('bleu') test

In [27]:
from transformers import BartTokenizer, BartForConditionalGeneration
import evaluate

tokenizer = BartTokenizer.from_pretrained('facebook/bart-large')

# 2) evaluate BLEU 로드
bleu = evaluate.load("bleu")          # 'blue' 오타 주의!

# 3) 래퍼 함수 정의 ─ 반드시 List[str] 반환
def hf_tokenizer(text: str):
    # special-token 제거 옵션(add_special_tokens=False)은 중요
    return tokenizer.tokenize(text)
# 예제 문장
preds = [
    "The quick brown fox jumps over the lazy dog", 
    "I love machine learning and natural language processing"
]

refs = [
    ["The quick brown fox jumps over the lazy dog"], 
    ["I love machine learning and natural language processing"]
]

# 4) 점수 계산
bleu_1 = bleu.compute(
    predictions = pred_string_list_previous,             # List[str]
    references  = target_string_list,              # List[List[str]]
    tokenizer   = hf_tokenizer,      # ← 커스텀 토크나이저 주입
    max_order   = 1,
    smooth      = True
)

bleu_2 = bleu.compute(
    predictions = pred_string_list_previous,             # List[str]
    references  = target_string_list,              # List[List[str]]
    tokenizer   = hf_tokenizer,      # ← 커스텀 토크나이저 주입
    max_order   = 2,
    smooth      = True
)

bleu_3 = bleu.compute(
    predictions = pred_string_list_previous,             # List[str]
    references  = target_string_list,              # List[List[str]]
    tokenizer   = hf_tokenizer,      # ← 커스텀 토크나이저 주입
    max_order   = 3,
    smooth      = True
)

bleu_4 = bleu.compute(
    predictions = pred_string_list_previous,             # List[str]
    references  = target_string_list,              # List[List[str]]
    tokenizer   = hf_tokenizer,      # ← 커스텀 토크나이저 주입
    max_order   = 4,
    smooth      = True
)


print([round(bleu_1["bleu"]*100,2),
       round(bleu_2["bleu"]*100,2),
       round(bleu_3["bleu"]*100,2),
       round(bleu_4["bleu"]*100,2)])


[45.0, 20.65, 12.87, 10.31]


In [78]:
target_string_list

['Everything its title implies, a standard-issue crime drama spat out from the Tinseltown assembly line.']

In [76]:
pred_string_list_previous

[' about about implies is from man,issue, drama. between of the depthsseltowns..']

In [82]:
b

[' about about implies is from man,issue, drama. between of the depthsseltowns..',
 '']

In [97]:
def preprocess(text: str) -> str:
    tokens = tokenizer.tokenize(text)
    return " ".join(tokens)          # 공백 단위 토큰열
preprocess("The quick brown fox jumps over the lazy dog")  # "The quick brown fox jumps over the lazy dog"

'The Ġquick Ġbrown Ġfox Ġjumps Ġover Ġthe Ġlazy Ġdog'

In [ ]:
import re
from rouge import Rouge
from collections import defaultdict

def valid_text(s: str) -> bool:
    """알파벳·숫자 토큰이 하나 이상 남는 문장인지 확인"""
    if re.search(r'\w', s) is None:
        return ''
    return s


a = []
a_1 = 'Everything its title implies, a standard-issue crime drama spat out from the Tinseltown assembly line.'
a.append(valid_text(a_1))  # True
a_2='A standard-issue crime drama spat out from the Tinseltown assembly line.'
a.append(valid_text(a_2))  # True
a_3 = '.'
a.append(valid_text(a_3))  # False
a_4 = ''
a.append(valid_text(a_4))  # False
a_5=' '
a.append(valid_text(a_5))  # False
print(a)

['Everything its title implies, a standard-issue crime drama spat out from the Tinseltown assembly line.', 'A standard-issue crime drama spat out from the Tinseltown assembly line.', '', '', '']


: 

In [107]:
a

['Everything its title implies, a standard-issue crime drama spat out from the Tinseltown assembly line.',
 'A standard-issue crime drama spat out from the Tinseltown assembly line.',
 'asdasd']

In [104]:
type(preds)

tuple

In [94]:
tokenizer.tokenize('.')

['.']

In [91]:
from rouge import Rouge
rouge = Rouge()
a = ['Everything its title implies, a standard-issue crime drama spat out from the Tinseltown assembly line.']
a.append('A standard-issue crime drama spat out from the Tinseltown assembly line.')
b=[' about about implies is from man,issue, drama. between of the depthsseltowns..']
b.append('a.')

try:
    rouge_scores = rouge.get_scores(b, a, avg = True, ignore_empty=True)
except ValueError as e:
    rouge_scores = 'Hypothesis is empty'
rouge_scores

{'rouge-1': {'f': 0.11538461294378703, 'p': 0.13636363636363635, 'r': 0.1},
 'rouge-2': {'f': 0.0, 'p': 0.0, 'r': 0.0},
 'rouge-l': {'f': 0.07999999760000007, 'p': 0.1, 'r': 0.06666666666666667}}

In [69]:
if "\t".strip():
    print("String is not empty")

AttributeError: 'NoneType' object has no attribute 'strip'

In [54]:
if len("   ".strip())==0: 
    print("Whitespace is stripped from the beginning and end of the string.")

Whitespace is stripped from the beginning and end of the string.


In [40]:
from rouge_score import rouge_scorer, scoring

preds = ["모델이 생성한 요약문 예시"]
refs  = ["사람이 작성한 참조 요약문 예시"]

score = rouge_scorer.RougeScorer(["rouge1"], use_stemmer=True)
agg    = scoring.BootstrapAggregator()     # 부트스트랩 95 % CI까지 계산

for pred, ref in zip(pred_string_list_previous, target_string_list):
    agg.add_scores(score.score(ref, pred))   # ref가 먼저!

summary = agg.aggregate()["rouge1"].mid        # .low, .high도 존재
print(
    f"ROUGE-1  R={summary.recall:.4f}  "
    f"P={summary.precision:.4f}  F={summary.fmeasure:.4f}"
)



INFO:absl:Using default tokenizer.


ROUGE-1  R=0.3125  P=0.4167  F=0.3571


In [30]:
metric_rouge = evaluate.load("rouge")
rouge   = metric_rouge.compute(
    predictions=pred_string_list_previous,
    references=target_string_list,
    use_stemmer=True,
    use_aggregator=False,
)
print(rouge)
rouge_scores = {}
for rouge_type in ["rouge1", "rouge2", "rougeL", "rougeLsum"]:
    score_mid = rouge[rouge_type]
    rouge_scores[rouge_type.upper()] = {
        "P":  score_mid['precision'] * 100,
        "R":  score_mid['recall'] * 100,
        "F1": score_mid['fmeasure'] * 100
    }

INFO:absl:Using default tokenizer.


{'rouge1': [0.35714285714285715], 'rouge2': [0.0], 'rougeL': [0.28571428571428575], 'rougeLsum': [0.28571428571428575]}


TypeError: list indices must be integers or slices, not str

In [18]:
bleu_1 = bleu.compute(
    predictions = pred_string_list_previous,             # List[str]
    references  = target_string_list,              # List[List[str]]
    tokenizer   = hf_tokenizer,      # ← 커스텀 토크나이저 주입
    max_order   = 1,
    smooth      = True
)

['The quick brown fox jumps over the lazy dog',
 'I love machine learning and natural language processing']

### Self-BLEU

In [ ]:
import collections
import math
import nltk
from nltk.util import ngrams
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# NLTK의 토크나이저 데이터를 처음 사용하는 경우 다운로드 필요
try:
    nltk.data.find('tokenizers/punkt')
except nltk.downloader.DownloadError:
    nltk.download('punkt')
    
def calculate_self_bleu(sentences: list[str]) -> float:
    """
    생성된 문장들 간의 유사도(Self-BLEU)를 계산합니다.
    
    :param sentences: 문장 리스트
    :return: Self-BLEU score
    """
    if len(sentences) < 2:
        return 0.0

    # 모든 문장을 미리 토큰화
    tokenized_sentences = [nltk.word_tokenize(s.lower()) for s in sentences]
    
    total_bleu_score = 0.0
    chencherry = SmoothingFunction() # 0점 방지를 위한 스무딩

    # 각 문장을 기준으로 나머지 문장들과 BLEU 점수 계산
    for i, hypothesis in enumerate(tokenized_sentences):
        references = tokenized_sentences[:i] + tokenized_sentences[i+1:]
        # sentence_bleu의 reference는 이중 리스트여야 함
        total_bleu_score += sentence_bleu(references, hypothesis, smoothing_function=chencherry.method1)
        
    return total_bleu_score / len(sentences)